### Etapa 1: Preparação do Ambiente e Planejamento

In [ ]:
# Bibliotecas de terceiros
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Módulos da biblioteca padrão do Python
import os
import re
import json
import random
from datetime import datetime, timedelta
from IPython.display import display

### Etapa 2: Ingestão de Dados (ETL - Extract)

##### RF01 – Criar ou Carregar o Dataset de Vendas

In [ ]:
# Carregar o CSV
df = pd.read_csv('../databases/vendas.csv')

### Etapa 3: Inspeção Estrutural (EDA Inicial)

##### RF02 – Inspecionar e Descrever os Dados

In [ ]:
# Função para inspecionar os dados
def inspecionar_dados(df):
    """Exibe as informações estruturais do DataFrame com tabelas renderizadas."""
    print("=== INSPEÇÃO INICIAL DO DATASET ===")
    
    # 1. Shape
    print(f'Shape: O dataframe tem [{df.shape[0]}] linhas e [{df.shape[1]}] colunas.\n')
    
    # 2. As 5 primeiras linhas (renderizadas)
    print("Primeiras 5 linhas:")
    display(df.head()) 
    
    # 3. As 5 últimas linhas (renderizadas)
    print("\nÚltimas 5 linhas:")
    display(df.tail())
    
    # 4. Tipos das colunas
    print("\nTipos das colunas:")
    display(df.dtypes.to_frame(name="Tipo de Dado"))
    
    # 5. Valores nulos
    print("\nValores nulos por coluna:")
    display(df.isnull().sum().to_frame(name="Qtd Nulos"))

In [ ]:
# Chamada da função (sem dar return no df para não duplicar no final)
inspecionar_dados(df)

In [ ]:
df.info()

In [ ]:
df.head(10)

### Etapa 4: Limpeza e Saneamento (ETL - Transform / Parte 1)

##### RF03 – Limpar e Tratar os Dados (datetime e regex)

In [ ]:
def limpar_dados(df):
    """
    Limpa e trata o DataFrame de vendas.
    Retorna: (df_limpo, relatorio), onde relatorio é um dicionario
    com as contagens de registros iniciais, removidos e finais.
    """

    # 1. remover espacos extras nas colunas de texto
    df_base = df
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].str.strip().upper()

    # 2. converter data_venda e descartar datas invalidas
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    df = df.dropna(subset=["data_venda"]) 

    # 3. descartar nulos em quantidade e preco_unitario
    df = df.dropna(subset=["quantidade", "preco_unitario"])

    # 4. ajustar os tipos numericos
    df["quantidade"] = pd.to_numeric(df["quantidade"], errors="coerce")
    df["preco_unitario"] = pd.to_numeric(df["preco_unitario"], errors="coerce")

    # 5. padronizar o nome do cliente com re.sub() e re.compile()
    df["cliente"] = df["cliente"].apply(
        lambda s: re.sub(r"[^A-Za-z0-9_]", "", str(s).strip())
    )

    # 6. montar e imprimir o relatorio de limpeza
    relatorio = {
        "registros_iniciais": len(df_base),
        "registros_removidos": len(df_base) - df.shape[0],
        "registros_finais": df.shape[0]
    }
    print("Relatório de Limpeza:")
    for chave, valor in relatorio.items():
        print(f"  {chave}: {valor}")
    
    return df, relatorio

In [ ]:
df_limpo = limpar_dados(df)[0]

In [ ]:
df_limpo.describe(include='all')

### Etapa 5: Engenharia de Atributos e Colunas Derivadas (ETL - Transform / Parte 2)

##### RF04 – Criar Colunas Derivadas com Transformações Condicionais

In [ ]:
# A partir do dataset limpo, crie no mínimo as seguintes colunas:

def criar_colunas_derivadas(df_limpo):
    """Cria colunas calculadas e temporais a partir do DataFrame limpo (RF04)."""
    df = df_limpo.copy()

    # 1. receita_total (Operação vetorizada)
    df["receita_total"] = df["quantidade"] * df["preco_unitario"]

    # 2. Extrações de data com o acessor .dt
    df["mes"] = df["data_venda"].dt.month
    df["trimestre"] = "Q" + df["data_venda"].dt.quarter.astype(str)
    df["ano"] = df["data_venda"].dt.year

    # 3. mes_nome (Mapeamento em português)
    mapa_meses = {
        1: "Janeiro",
        2: "Fevereiro",
        3: "Março",
        4: "Abril",
        5: "Maio",
        6: "Junho",
        7: "Julho",
        8: "Agosto",
        9: "Setembro",
        10: "Outubro",
        11: "Novembro",
        12: "Dezembro",
    }
    df["mes_nome"] = df["mes"].map(mapa_meses)

    # 4. faixa_receita_item (Transformação condicional vetorizada)
    condicoes = [
        df["receita_total"] < 500,
        (df["receita_total"] >= 500) & (df["receita_total"] < 5000),
        df["receita_total"] >= 5000,
    ]
    faixas = ["Baixo Valor", "Medio Valor", "Alto Valor"]

    df["faixa_receita_item"] = np.select(
        condicoes, faixas, default="Nao Classificado"
    )

    return df

In [ ]:
# Execução no seu notebook / script:
df_processado = criar_colunas_derivadas(df_limpo)

In [ ]:
df_processado.head(10)

### Etapa 6: Análise Exploratória, Agregações e Segmentação (EDA Avançada)

##### RF05 – Calcular Métricas Agregadas com groupby

In [ ]:
def calcular_metricas(df):
    """Calcula métricas agregadas do dataset separando cada dimensão passo a passo.

    Retorna um dicionário contendo DataFrames formatados.
    """
    metricas = {}

    # -------------------------------------------------------------------------
    # 1. MÉTRICAS POR MÊS
    # -------------------------------------------------------------------------
    # Agrupamos por 'mes' e somamos a 'receita_total' e a 'quantidade'
    por_mes = df.groupby(["mes", "mes_nome"])[["receita_total", "quantidade"]].sum()

    # Contamos quantas vendas/linhas existem para cada mês
    por_mes["n_vendas"] = df.groupby("mes")["id_venda"].count().values
    # Transformamos o índice 'mes' de volta em uma coluna normal
    por_mes = por_mes.reset_index().sort_values("mes")

    # Guardamos no dicionário
    metricas["por_mes"] = por_mes

    # -------------------------------------------------------------------------
    # 2. TOP 5 PRODUTOS POR RECEITA
    # -------------------------------------------------------------------------
    # Agrupamos por 'produto' e somamos a 'receita_total'
    top_produtos = df.groupby("produto")[["receita_total"]].sum()

    # Ordenamos do maior para o menor (ascending=False)
    top_produtos = top_produtos.sort_values(by="receita_total", ascending=False)

    # Pegamos apenas os 5 primeiros registros
    top_produtos = top_produtos.head(5)

    # Transformamos o índice em coluna
    top_produtos = top_produtos.reset_index()

    # Guardamos no dicionário
    metricas["top_produtos"] = top_produtos

    # -------------------------------------------------------------------------
    # 3. RECEITA POR CATEGORIA
    # -------------------------------------------------------------------------
    # Agrupamos por 'categoria' e somamos a 'receita_total'
    por_categoria = df.groupby("categoria")[["receita_total"]].sum()

    # Ordenamos da maior para a menor receita
    por_categoria = por_categoria.sort_values(
        by="receita_total", ascending=False
    )

    # Transformamos o índice em coluna
    por_categoria = por_categoria.reset_index()

    # Guardamos no dicionário
    metricas["por_categoria"] = por_categoria

    # -------------------------------------------------------------------------
    # 4. RECEITA TOTAL E TICKET MÉDIO POR REGIÃO
    # -------------------------------------------------------------------------
    # Agrupamos por 'regiao' e calculamos a SOMA da receita
    por_regiao = df.groupby("regiao")[["receita_total"]].sum()

    # Criamos a coluna 'ticket_medio' calculando a MÉDIA da receita
    por_regiao["ticket_medio"] = df.groupby("regiao")["receita_total"].mean()

    # Ordenamos da maior para a menor receita total
    por_regiao = por_regiao.sort_values(by="receita_total", ascending=False)

    # Transformamos o índice em coluna
    por_regiao = por_regiao.reset_index()

    # Guardamos no dicionário
    metricas["por_regiao"] = por_regiao

    return metricas


def exibir_metricas(metricas):
    """Exibe no console os resultados de forma legível e sem os índices do Pandas."""
    print("\n=== POR MÊS ===")
    print(metricas["por_mes"].to_string(index=False))

    print("\n=== TOP PRODUTOS ===")
    print(metricas["top_produtos"].to_string(index=False))

    print("\n=== POR CATEGORIA ===")
    print(metricas["por_categoria"].to_string(index=False))

    print("\n=== POR REGIÃO ===")
    print(metricas["por_regiao"].to_string(index=False))

In [ ]:
exibir_metricas(calcular_metricas(df_processado))

##### RF06 – Segmentar Clientes por Nível de Gasto

In [ ]:
# 1. Passo: Criamos uma função de apoio em Python bem clara
def classificar_valor(valor):
    """Recebe um número e retorna a categoria correspondente."""
    if valor < 5000:
        return "Bronze"
    elif valor <= 15000:
        return "Prata"
    else:
        return "Ouro"


def segmentar_clientes(df):
    """Agrupa por cliente, calcula o total gasto e aplica a classificação

    em Bronze, Prata e Ouro.
    """
    # Passo A: Agrupar por cliente e somar o total gasto
    clientes = df.groupby("cliente")[["receita_total"]].sum().reset_index()

    # Renomeamos para o nome solicitado no requisito ('total_gasto')
    clientes = clientes.rename(columns={"receita_total": "total_gasto"})

    # Passo B: Aplicar a função lambda com .apply() usando a nossa função de apoio
    clientes["segmento"] = clientes["total_gasto"].apply(
        lambda x: classificar_valor(x)
    )

    # Passo C: Ordenar do cliente que mais gastou para o que menos gastou
    clientes = clientes.sort_values(by="total_gasto", ascending=False)

    return clientes


def exibir_analise_clientes(df_clientes):
    """Exibe no console os 10 maiores clientes e a distribuição de segmentos."""
    print("\n=== TOP 10 CLIENTES ===")
    # .head(10) pega as 10 primeiras linhas da tabela ordenada
    top_10 = df_clientes.head(10)
    print(top_10.to_string(index=False))

    print("\n=== DISTRIBUIÇÃO DE CLIENTES POR SEGMENTO ===")
    # .value_counts() conta quantos clientes caíram em cada segmento
    distribuicao = df_clientes["segmento"].value_counts()
    print(distribuicao.to_string())

In [ ]:

exibir_analise_clientes(segmentar_clientes(df_processado))


##### RF07 – Operações Numéricas com NumPy

In [ ]:
def calcular_estatisticas_numpy(df):
    """Realiza operações numéricas vetorizadas com NumPy sobre a receita.

    Demonstra conversão para array, agregação, broadcasting e filtragem
    booleana.
    """
    # -------------------------------------------------------------------------
    # 1. CONVERSÃO PARA ARRAY NUMPY
    # -------------------------------------------------------------------------
    # Extrai os dados da coluna 'receita_total' do Pandas para um array NumPy
    receitas = df["receita_total"].to_numpy()

    # -------------------------------------------------------------------------
    # 2. FUNÇÕES DE AGREGAÇÃO DO NUMPY
    # -------------------------------------------------------------------------
    # Calculamos quatro estatísticas básicas diretamente no array
    media = np.mean(receitas)
    mediana = np.median(receitas)
    desvio_padrao = np.std(receitas)
    soma_total = np.sum(receitas)
    valor_minimo = np.min(receitas)
    valor_maximo = np.max(receitas)

    # -------------------------------------------------------------------------
    # 3. BROADCASTING E OPERAÇÕES VETORIZADAS (Escalonamento 0 a 1)
    # -------------------------------------------------------------------------
    # Fórmula Min-Max: (x - min) / (max - min)
    # O NumPy faz a subtração e a divisão elemento por elemento automaticamente,
    # sem a necessidade de um laço 'for'.
    amplitude = valor_maximo - valor_minimo
    receitas_escalonadas = (receitas - valor_minimo) / amplitude

    # -------------------------------------------------------------------------
    # 4. FILTRAGEM BOOLEANA
    # -------------------------------------------------------------------------
    # Cria uma "máscara" de Verdadeiro/Falso e filtra apenas os valores acima da média
    vendas_acima_da_media = receitas[receitas > media]
    quantidade_acima_da_media = len(vendas_acima_da_media)

    # -------------------------------------------------------------------------
    # 5. ESTRUTURAÇÃO DO RETORNO (Dicionário)
    # -------------------------------------------------------------------------
    estatisticas = {
        "media": media,
        "mediana": mediana,
        "desvio_padrao": desvio_padrao,
        "soma_total": soma_total,
        "valor_minimo": valor_minimo,
        "valor_maximo": valor_maximo,
        "quantidade_acima_media": quantidade_acima_da_media,
        "array_escalonado": receitas_escalonadas,
    }

    return estatisticas


def exibir_estatisticas_numpy(estatisticas):
    """Exibe os resultados das operações do NumPy de forma legível no console."""
    print("\n=== ESTATÍSTICAS NUMPY (RECEITA TOTAL) ===")
    print(f"Média: R$ {estatisticas['media']:.2f}")
    print(f"Mediana: R$ {estatisticas['mediana']:.2f}")
    print(f"Desvio Padrão: R$ {estatisticas['desvio_padrao']:.2f}")
    print(f"Soma Total: R$ {estatisticas['soma_total']:.2f}")
    print(f"Valor Mínimo: R$ {estatisticas['valor_minimo']:.2f}")
    print(f"Valor Máximo: R$ {estatisticas['valor_maximo']:.2f}")
    print(
        f"Vendas acima da média: {estatisticas['quantidade_acima_media']} transações"
    )

    print("\n=== BROADCASTING: PRIMEIRAS 5 RECEITAS ESCALONADAS (0 A 1) ===")
    print(estatisticas["array_escalonado"][:5])


In [ ]:
exibir_estatisticas_numpy(calcular_estatisticas_numpy(df_processado))

In [ ]:
def executar_e_exibir(funcao_calculo, funcao_exibicao, dados):
    """Função de Ordem Superior: ela recebe as duas funções operacionais

    (cálculo e exibição) e os dados brutos.

    Ela executa o cálculo e repassa o resultado diretamente para a exibição.
    """
    # Passo 1: Executa a função de cálculo enviada
    resultado = funcao_calculo(dados)

    # Passo 2: Passa o resultado para a função de exibição enviada
    funcao_exibicao(resultado)


In [ ]:
# -------------------------------------------------------------------------
# 1. FUNÇÃO DE ORDEM SUPERIOR: EXECUTAR E EXIBIR
# -------------------------------------------------------------------------
executar_e_exibir(
    calcular_estatisticas_numpy,
    exibir_estatisticas_numpy,
    df_processado,
)

### Etapa 7: Visualização de Dados e Geração de Dashboards

##### RF08 – Criar Visualizações com Matplotlib e Seaborn

In [ ]:
# =============================================================================
# RF08 – Criar Visualizações com Matplotlib e Seaborn
# =============================================================================

def gerar_visualizacoes(df_processado, metricas):
    """
    Gera e salva 4 gráficos analíticos para a base de vendas:
    1. Vendas por Mês (Gráfico de Linha)
    2. Receita por Categoria (Gráfico de Barras)
    3. Distribuição das Vendas (Histograma com KDE)
    4. Top 5 Produtos por Receita (Gráfico de Barras Horizontais)
    """
    # Configuração de estilo global do Seaborn e Matplotlib
    sns.set_theme(style="whitegrid")
    plt.rcParams["font.family"] = "sans-serif"

    # -------------------------------------------------------------------------
    # 1. Vendas por Mês (Linha)
    # -------------------------------------------------------------------------
    plt.figure(figsize=(10, 5))
    df_mes = metricas["por_mes"]
    
    sns.lineplot(
        data=df_mes, 
        x="mes_nome", 
        y="receita_total", 
        marker="o", 
        color="#1f77b4", 
        linewidth=2.5
    )
    plt.title("Evolução Mensal da Receita Total", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Mês", fontsize=11)
    plt.ylabel("Receita (R$)", fontsize=11)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("../outputs/graficos/vendas_por_mes.png", dpi=300)
    plt.show()

    # -------------------------------------------------------------------------
    # 2. Receita por Categoria (Barras)
    # -------------------------------------------------------------------------
    plt.figure(figsize=(8, 5))
    df_cat = metricas["por_categoria"]
    
    sns.barplot(
        data=df_cat, 
        x="categoria", 
        y="receita_total", 
        palette="Blues_r"
    )
    plt.title("Receita Total por Categoria de Produto", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Categoria", fontsize=11)
    plt.ylabel("Receita (R$)", fontsize=11)
    plt.tight_layout()
    plt.savefig("../outputs/graficos/receita_por_categoria.png", dpi=300)
    plt.show()

    # -------------------------------------------------------------------------
    # 3. Distribuição do Valor das Vendas (Histograma / KDE)
    # -------------------------------------------------------------------------
    plt.figure(figsize=(9, 5))
    
    sns.histplot(
        df_processado["receita_total"], 
        kde=True, 
        color="#2ca02c", 
        bins=20
    )
    plt.title("Distribuição do Valor das Vendas (Receita por Transação)", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Valor da Venda (R$)", fontsize=11)
    plt.ylabel("Frequência", fontsize=11)
    plt.tight_layout()
    plt.savefig("../outputs/graficos/distribuicao_vendas.png", dpi=300)
    plt.show()

    # -------------------------------------------------------------------------
    # 4. Top 5 Produtos por Receita (Barras Horizontais)
    # -------------------------------------------------------------------------
    plt.figure(figsize=(9, 5))
    df_top = metricas["top_produtos"]
    
    sns.barplot(
        data=df_top, 
        x="receita_total", 
        y="produto", 
        palette="viridis"
    )
    plt.title("Top 5 Produtos por Receita Total", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Receita (R$)", fontsize=11)
    plt.ylabel("Produto", fontsize=11)
    plt.tight_layout()
    plt.savefig("../outputs/graficos/top_5_produtos.png", dpi=300)
    plt.show()

    print("\nVisualizações geradas e salvas em PNG com sucesso!")

In [ ]:
# Célula isolada de teste para o RF08
metricas = calcular_metricas(df_processado)
gerar_visualizacoes(df_processado, metricas)